<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aibizx/learn-ml-algorithms-2017-notebooks/blob/main/03-gradient-descent.ipynb)

_Part of the [AI/Biz books](https://www.ai.biz/books/learn-ml-algorithms-2017/) collection._

# Chapter 3 — Gradient Descent, Seen Clearly

Companion to [the chapter](https://www.ai.biz/books/learn-ml-algorithms-2017/gradient-descent-seen-clearly/).

The same loss as chapter 2, found this time by walking downhill instead of solving an equation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def make_deliveries(n, noise=4.0, seed=0):
    rng = np.random.default_rng(seed)
    km = rng.uniform(0.5, 8.0, n)
    items = rng.integers(1, 7, n)
    minutes = 8 + 4.0 * km + 1.5 * items + rng.normal(0, noise, n)
    return km, items, minutes

km, items, minutes = make_deliveries(200, seed=1)
X = np.column_stack([np.ones(len(km)), km, items])
y = minutes

def loss(w):
    return np.mean((X @ w - y) ** 2)

def gradient(w):
    return 2 / len(y) * X.T @ (X @ w - y)

w_exact = np.linalg.solve(X.T @ X, X.T @ y)
print('the answer we are walking towards:', np.round(w_exact, 3), f' loss {loss(w_exact):.3f}')

## 1. Checking the gradient by hand

In [ ]:
def numerical_gradient(f, w, h=1e-6):
    """Nudge each weight a little and see how much the loss moves."""
    g = np.zeros_like(w)
    for i in range(len(w)):
        step = np.zeros_like(w); step[i] = h
        g[i] = (f(w + step) - f(w - step)) / (2 * h)
    return g

w0 = np.zeros(3)
print('formula  :', np.round(gradient(w0), 4))
print('numerical:', np.round(numerical_gradient(loss, w0), 4))

## 2. Walking downhill

In [ ]:
def gradient_descent(w, rate, steps, every=None):
    history = [loss(w)]
    for t in range(1, steps + 1):
        w = w - rate * gradient(w)
        history.append(loss(w))
        if every and t % every == 0:
            print(f'  step {t:>5}  loss {history[-1]:9.3f}  w = {np.round(w, 3)}')
    return w, history

print('learning rate 0.01')
w_gd, hist = gradient_descent(np.zeros(3), rate=0.01, steps=2000, every=400)
print('exact:', np.round(w_exact, 3))

## 3. Three learning rates

In [ ]:
for rate in [0.001, 0.01, 0.05]:
    w_r, h = gradient_descent(np.zeros(3), rate=rate, steps=200)
    tail = h[-1]
    print(f'rate {rate:<6} after 200 steps: ' + (f'loss {tail:.1f}' if tail < 1e6 else 'loss overflowed, the walk diverged'))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for rate in [0.001, 0.01, 0.03]:
    _, h = gradient_descent(np.zeros(3), rate=rate, steps=300)
    ax.plot(h, label=f'rate {rate}')
ax.axhline(loss(w_exact), color='black', lw=1, ls='--', label='exact minimum')
ax.set_ylim(0, 200); ax.set_xlabel('step'); ax.set_ylabel('mean squared error'); ax.legend()
plt.show()

## 4. Why the scale of a feature matters

In [ ]:
metres = km * 1000
X_m = np.column_stack([np.ones(len(km)), metres, items])

def loss_m(w): return np.mean((X_m @ w - y) ** 2)
def grad_m(w): return 2 / len(y) * X_m.T @ (X_m @ w - y)

w = np.zeros(3)
for t in range(5):
    w = w - 0.01 * grad_m(w)
    print(f'step {t + 1}: loss {loss_m(w):.3g}')

In [ ]:
# The curvature of the bowl in each direction is set by the eigenvalues of XᵀX / n
for name, M in [('km', X), ('metres', X_m)]:
    eig = np.linalg.eigvalsh(M.T @ M / len(y))
    print(f'{name:<7} curvature: smallest {eig.min():10.4f}, largest {eig.max():14.1f}, ratio {eig.max() / eig.min():12,.0f}')

In [ ]:
def standardise(col):
    return (col - col.mean()) / col.std()

X_s = np.column_stack([np.ones(len(km)), standardise(metres), standardise(items)])

def loss_s(w): return np.mean((X_s @ w - y) ** 2)
def grad_s(w): return 2 / len(y) * X_s.T @ (X_s @ w - y)

w = np.zeros(3)
for t in range(1, 101):
    w = w - 0.1 * grad_s(w)
    if t in (1, 10, 50, 100):
        print(f'step {t:>3}: loss {loss_s(w):8.3f}')
eig = np.linalg.eigvalsh(X_s.T @ X_s / len(y))
print(f'standardised curvature ratio: {eig.max() / eig.min():.2f}')

## 5. Stochastic gradient descent

In [ ]:
rng = np.random.default_rng(0)

def sgd(X, y, rate, epochs, batch):
    w = np.zeros(X.shape[1])
    n = len(y)
    for epoch in range(epochs):
        order = rng.permutation(n)
        for start in range(0, n, batch):
            idx = order[start:start + batch]
            g = 2 / len(idx) * X[idx].T @ (X[idx] @ w - y[idx])
            w = w - rate * g
    return w

for batch in [1, 20, 200]:
    w_b = sgd(X_s, y, rate=0.05, epochs=20, batch=batch)
    updates = 20 * int(np.ceil(200 / batch))
    print(f'batch {batch:>3}: {updates:>5} updates, loss {loss_s(w_b):8.3f}')
print(f'exact minimum on standardised features: {loss_s(np.linalg.solve(X_s.T @ X_s, X_s.T @ y)):.3f}')

## 6. When to stop

In [ ]:
w = np.zeros(3)
previous = loss_s(w)
for t in range(1, 10001):
    g = grad_s(w)
    w = w - 0.1 * g
    current = loss_s(w)
    if np.linalg.norm(g) < 1e-4 or previous - current < 1e-9:
        print(f'stopped at step {t}: gradient norm {np.linalg.norm(g):.2e}, loss change {previous - current:.2e}')
        break
    previous = current

## 7. Not every loss is a bowl

In [ ]:
def bumpy(x):
    return x**2 / 20 + np.sin(2 * x)

def bumpy_grad(x):
    return x / 10 + 2 * np.cos(2 * x)

for start in [-5.0, -2.0, 0.5, 4.0]:
    x = start
    for _ in range(500):
        x = x - 0.05 * bumpy_grad(x)
    print(f'start {start:5.1f} -> ends at x = {x:6.2f}, value {bumpy(x):6.2f}')

In [ ]:
grid = np.linspace(-7, 7, 400)
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(grid, bumpy(grid))
for start in [-5.0, -2.0, 0.5, 4.0]:
    x = start
    for _ in range(500):
        x = x - 0.05 * bumpy_grad(x)
    ax.plot([start, x], [bumpy(start), bumpy(x)], 'o-', ms=5, alpha=0.7)
ax.set_xlabel('x'); ax.set_ylabel('loss'); ax.set_title('four starts, three different endings')
plt.show()

## 8. Momentum

In [ ]:
def descend(rate, steps, momentum=0.0):
    w, v = np.zeros(3), np.zeros(3)
    for _ in range(steps):
        v = momentum * v - rate * gradient(w)     # keep some of last step's direction
        w = w + v
    return loss(w)

print(f'{"steps":>6} {"plain":>10} {"momentum 0.9":>13}')
for steps in [50, 100, 200, 500]:
    print(f'{steps:>6} {descend(0.01, steps):10.3f} {descend(0.01, steps, 0.9):13.3f}')
print(f'minimum {loss(w_exact):.3f}')

## 9. The one line you would ship

In [ ]:
from sklearn.linear_model import SGDRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

features = np.column_stack([km, items])
sgd_model = make_pipeline(StandardScaler(),
                          SGDRegressor(learning_rate='constant', eta0=0.01, max_iter=200,
                                       tol=None, penalty=None, random_state=0))
sgd_model.fit(features, minutes)
reg = sgd_model[-1]
print(f'SGDRegressor loss on training data: {np.mean((sgd_model.predict(features) - minutes) ** 2):.3f}')
print(f'exact minimum                     : {loss(w_exact):.3f}')

## Try it yourself

1. In section 3, find the largest learning rate that still converges. Compare it with 2 divided by the largest curvature printed in section 4.
2. Run section 5 with `batch=1` and `epochs=1`. One pass through the data, one row at a time: how close does it get?
3. In section 7, add momentum to the walk. Do any of the four starts now escape their valley?